# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB as part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

**Design.** The agent is built on the course library (`lib.state_machine`, `lib.tooling`, `lib.llm`, `lib.memory`).
Instead of letting the LLM improvise the order of tool calls, the workflow is an explicit state machine, so
*retrieve -> evaluate -> web search only if needed -> answer* always happens in that order:

```
entry -> understand -> retrieve -> evaluate --useful & confidence >= medium--> answer -> remember -> end
                                      |                                         ^
                                      +--- not good enough --> web_search ------+
```

| Step | What happens | Tool |
|---|---|---|
| `understand` | Uses the session history to turn follow-ups ("Who published it?") into standalone questions | - |
| `retrieve` | Semantic search over the local dataset (and long-term memory) | `retrieve_game`, `recall_memory` |
| `evaluate` | LLM judge decides whether the results are enough, with a confidence level | `evaluate_retrieval` |
| `web_search` | Fallback, only when the evaluation fails | `game_web_search` |
| `answer` | Structured answer + confidence + citations | - |
| `remember` | Web findings are stored in long-term memory | - |

### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import hashlib
from datetime import datetime, timezone
from typing import List, Literal, Optional, TypedDict

import chromadb
from chromadb.utils import embedding_functions
from pydantic import BaseModel, Field
from tavily import TavilyClient
from dotenv import load_dotenv, find_dotenv

from lib.llm import LLM
from lib.messages import SystemMessage, UserMessage
from lib.tooling import tool
from lib.state_machine import StateMachine, Step, EntryPoint, Termination, Run
from lib.memory import ShortTermMemory

In [ ]:
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"

MODEL = "gpt-4o-mini"

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# Same path, collection name and embedding function as Part 1
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay", embedding_function=embedding_fn)

# Long-term memory (stand-out): a second collection in the same persistent database
memory_collection = chroma_client.get_or_create_collection("udaplay_memory", embedding_function=embedding_fn)

print("Games in vector DB:", collection.count(), "| Long-term memories:", memory_collection.count())

#### Helpers
- `llm_structured`: calls `lib.llm.LLM` with a Pydantic model as `response_format` and parses the reply (structured output).
- `call_tool`: runs a tool and prints the call, so every tool use is visible in the notebook output.

In [ ]:
def llm_structured(system: str, user: str, schema):
    """Ask the LLM (lib.llm.LLM) and get the reply back as an instance of the Pydantic model `schema`."""
    llm = LLM(model=MODEL, temperature=0.0)
    response = llm.invoke(
        [SystemMessage(content=system), UserMessage(content=user)],
        response_format=schema,
    )
    return schema.model_validate_json(response.content)


def call_tool(t, **kwargs):
    """Run a lib Tool and log the call."""
    shown = ", ".join(f"{k}={repr(v)[:70]}" for k, v in kwargs.items())
    print(f"   [TOOL CALL] {t.name}({shown})")
    return t(**kwargs)

### Tools

Build at least 3 tools:
- `retrieve_game`: To search the vector DB
- `evaluate_retrieval`: To assess the retrieval performance
- `game_web_search`: If no good, search the web
- (extra) `recall_memory`: search long-term memory of earlier web findings

#### Retrieve Game Tool

In [ ]:
@tool
def retrieve_game(query: str, n_results: int = 3) -> List[dict]:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - query: a question about game industry.

    You'll receive results as list. Each element contains:
    - Platform: like Game Boy, PlayStation 5, Xbox 360...
    - Name: Name of the Game
    - YearOfRelease: Year when that game was released for that platform
    - Description: Additional details about the game
    (plus Genre, Publisher, id and distance - a lower distance means a closer match)
    """
    res = collection.query(query_texts=[query], n_results=n_results)
    docs = []
    for i in range(len(res["ids"][0])):
        item = dict(res["metadatas"][0][i])
        item["id"] = res["ids"][0][i]
        item["distance"] = round(res["distances"][0][i], 3)
        item["source"] = "local_dataset"
        docs.append(item)
    return docs


# quick test (calls the Tool object directly)
for d in retrieve_game("Who published Grand Theft Auto: San Andreas?"):
    print("  ->", d["Name"], "|", d["Platform"], "|", d["YearOfRelease"], "|", d["Publisher"], "| dist", d["distance"])
print(retrieve_game)

#### Evaluate Retrieval Tool

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the documents are enough to answer the question")
    confidence: Literal["high", "medium", "low"] = Field(description="Confidence that the docs answer the question")
    description: str = Field(description="Detailed explanation of the evaluation so an action can be taken")


@tool
def evaluate_retrieval(question: str, retrieved_docs: List[dict]) -> EvaluationReport:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
    - useful: whether the documents are useful to answer the question
    - confidence: high / medium / low
    - description: description about the evaluation result
    """
    system = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        "Give a detailed explanation, so it's possible to take an action to accept it or not. "
        "Be strict: the documents must explicitly contain the specific fact asked "
        "(the right game, the right platform, the right attribute). "
        "A similar game, a sequel, or a different platform does NOT count. "
        "Questions about current or recent events, or about games/companies that do not appear in the documents, "
        "are NOT answerable from the documents. "
        "If the documents clearly contain the needed fact, confidence is 'high'. "
        "If they only partially cover it, use 'medium' or 'low'."
    )
    user = f"Question: {question}\n\nRetrieved documents:\n{json.dumps(retrieved_docs, indent=2, default=str)}"
    return llm_structured(system, user, EvaluationReport)

#### Game Web Search Tool

In [ ]:
@tool
def game_web_search(question: str) -> dict:
    """
    Web search: searches the web with Tavily when the vector DB results aren't good enough
    args:
    - question: a question about game industry.

    Returns the search results, including each page's URL, so the answer can cite its source.
    """
    res = tavily_client.search(query=question, max_results=4, include_answer=True)
    return {
        "tavily_answer": res.get("answer"),
        "results": [
            {"title": r.get("title"), "url": r.get("url"), "content": (r.get("content") or "")[:700]}
            for r in res.get("results", [])
        ],
    }

#### Long-term memory tool (Stand Out)
Searches what the agent learned from earlier web searches (kept in the persistent `udaplay_memory` collection).

In [ ]:
@tool
def recall_memory(query: str, n_results: int = 2) -> List[dict]:
    """
    Long-term memory: searches facts learned from earlier web searches.
    args:
    - query: a question about game industry.
    Returns remembered question/answer pairs together with the URL they came from.
    """
    total = memory_collection.count()
    if total == 0:
        return []
    res = memory_collection.query(query_texts=[query], n_results=min(n_results, total))
    out = []
    for i in range(len(res["ids"][0])):
        m = res["metadatas"][0][i]
        out.append({
            "source": "long_term_memory",
            "question": m["question"], "answer": m["answer"], "url": m["url"],
            "saved_at": m["saved_at"], "distance": round(res["distances"][0][i], 3),
        })
    return out


def remember(question: str, answer: str, url: str):
    """Save a web-sourced answer in long-term memory (upsert keyed by the question)."""
    mem_id = hashlib.md5(question.lower().strip().encode()).hexdigest()
    memory_collection.upsert(
        ids=[mem_id],
        documents=[f"Q: {question}\nA: {answer}"],
        metadatas=[{"question": question, "answer": answer, "url": url,
                    "saved_at": datetime.now(timezone.utc).strftime("%Y-%m-%d")}],
    )

### Agent

- **State machine:** built with `lib.state_machine` (`StateMachine`, `Step`, `EntryPoint`, `Termination`). Each step is a node, the branch after `evaluate` is a conditional transition.
- **Conversation state:** every finished `Run` is stored in `lib.memory.ShortTermMemory` under a session id. The next question in the same session rebuilds its history from those runs.
- **Structured output:** the final answer is an `AgentAnswer` (answer, confidence, source type, sources).

In [ ]:
class AgentAnswer(BaseModel):
    answer: str = Field(description="Clear natural-language answer to the question")
    confidence: Literal["high", "medium", "low"]
    source_type: Literal["local_dataset", "web", "long_term_memory", "both"]
    sources: List[str] = Field(description="Citations: web URLs, or 'Local dataset: <Game> (<Platform>, id <id>)'")


class Rewrite(BaseModel):
    standalone_question: str


# State carried through the workflow (lib.state_machine only keeps fields declared here)
class UdaState(TypedDict):
    session_id: str
    question: str
    history: List[dict]
    standalone: str
    local_docs: List[dict]
    memory_docs: List[dict]
    evaluation: Optional[dict]
    web: Optional[dict]
    answer: Optional[dict]

In [ ]:
class UdaPlayAgent:
    def __init__(self, long_term_memory: bool = True):
        self.long_term_memory = long_term_memory
        self.memory = ShortTermMemory()          # conversation state: one list of Runs per session
        self.workflow = self._create_state_machine()

    # ------------------------------------------------------------------ workflow
    def _create_state_machine(self) -> StateMachine[UdaState]:
        machine = StateMachine[UdaState](UdaState)

        entry = EntryPoint[UdaState]()
        understand = Step[UdaState]("understand", self.s_understand)
        retrieve = Step[UdaState]("retrieve", self.s_retrieve)
        evaluate = Step[UdaState]("evaluate", self.s_evaluate)
        web_search = Step[UdaState]("web_search", self.s_web_search)
        answer = Step[UdaState]("answer", self.s_answer)
        remember_step = Step[UdaState]("remember", self.s_remember)
        end = Termination[UdaState]()

        machine.add_steps([entry, understand, retrieve, evaluate, web_search, answer, remember_step, end])

        machine.connect(entry, understand)
        machine.connect(understand, retrieve)
        machine.connect(retrieve, evaluate)

        def route_after_evaluate(state: UdaState):
            """Internal knowledge is good enough -> answer; otherwise fall back to the web."""
            e = state["evaluation"]
            if e["useful"] and e["confidence"] != "low":
                print("   decision: internal knowledge is good enough -> answer")
                return answer
            print("   decision: internal knowledge is NOT good enough -> web search")
            return web_search

        machine.connect(evaluate, [answer, web_search], route_after_evaluate)
        machine.connect(web_search, answer)
        machine.connect(answer, remember_step)
        machine.connect(remember_step, end)
        return machine

    # ------------------------------------------------------------------ steps
    def s_understand(self, state: UdaState) -> dict:
        print("\n>> STEP: understand")
        standalone = state["question"]
        if state["history"]:
            system = ("Rewrite the user's latest message as a standalone question about video games, "
                      "resolving pronouns and references (it, that game, its sequel...) using the conversation. "
                      "If it is already standalone, return it unchanged.")
            user = (f"Conversation so far:\n{json.dumps(state['history'][-6:], indent=2)}\n\n"
                    f"Latest message: {state['question']}")
            standalone = llm_structured(system, user, Rewrite).standalone_question
        print(f"   standalone question: {standalone}")
        return {"standalone": standalone}

    def s_retrieve(self, state: UdaState) -> dict:
        print("\n>> STEP: retrieve")
        local_docs = call_tool(retrieve_game, query=state["standalone"])
        for d in local_docs:
            print(f"     local : {d['Name']} | {d['Platform']} | {d['YearOfRelease']} | dist {d['distance']}")
        memory_docs = []
        if self.long_term_memory:
            memory_docs = call_tool(recall_memory, query=state["standalone"])
            for m in memory_docs:
                print(f"     memory: {m['question']} | dist {m['distance']}")
        return {"local_docs": local_docs, "memory_docs": memory_docs}

    def s_evaluate(self, state: UdaState) -> dict:
        print("\n>> STEP: evaluate")
        report = call_tool(evaluate_retrieval, question=state["standalone"],
                           retrieved_docs=state["local_docs"] + state["memory_docs"])
        print(f"   useful={report.useful} | confidence={report.confidence}")
        print(f"   reason: {report.description}")
        return {"evaluation": report.model_dump()}

    def s_web_search(self, state: UdaState) -> dict:
        print("\n>> STEP: web_search")
        try:
            web = call_tool(game_web_search, question=state["standalone"])
            for r in web["results"]:
                print(f"     web: {r['title'][:60]} | {r['url']}")
        except Exception as ex:
            print(f"   web search failed: {ex}")
            web = None
        return {"web": web}

    def s_answer(self, state: UdaState) -> dict:
        print("\n>> STEP: answer")
        evidence = {
            "local_dataset_results": state["local_docs"],
            "long_term_memory": state["memory_docs"],
            "retrieval_evaluation": state["evaluation"],
        }
        if state["web"]:
            evidence["web_search"] = state["web"]
        system = (
            "You are UdaPlay, an AI research assistant for the video game industry. "
            "Answer ONLY from the evidence provided; never invent facts. "
            "Use local dataset results when the evaluation says they are useful; otherwise rely on web results. "
            "If you combine sources, set source_type to 'both'. "
            "If the evidence is insufficient, say so plainly and set confidence to 'low'. "
            "Citations: for local data use 'Local dataset: <Game> (<Platform>, id <id>)'; "
            "for web or memory facts use the exact URL from the evidence. "
            "Be concise and natural (1-3 sentences)."
        )
        user = (f"Conversation so far:\n{json.dumps(state['history'][-6:])}\n\n"
                f"Question: {state['standalone']}\n\nEvidence:\n{json.dumps(evidence, indent=2, default=str)}")
        ans = llm_structured(system, user, AgentAnswer)

        # guardrail: only keep URLs that really came from the evidence
        known = [r["url"] for r in (state["web"] or {}).get("results", [])] + [m["url"] for m in state["memory_docs"]]
        ans.sources = [s for s in ans.sources if (not s.startswith("http")) or s in known]
        if ans.source_type in ("web", "both") and known and not any(s.startswith("http") for s in ans.sources):
            ans.sources.append(known[0])
        return {"answer": ans.model_dump()}

    def s_remember(self, state: UdaState) -> dict:
        print("\n>> STEP: remember")
        ans = state["answer"]
        web_urls = [r["url"] for r in (state["web"] or {}).get("results", [])]
        cited_web = [s for s in ans["sources"] if s in web_urls]
        if self.long_term_memory and cited_web and ans["confidence"] != "low":
            remember(state["standalone"], ans["answer"], cited_web[0])
            print(f"   saved to long-term memory ({memory_collection.count()} memories)")
        else:
            print("   nothing to save")
        return {}

    # ------------------------------------------------------------------ public API
    def _history(self, session_id: str) -> List[dict]:
        """Rebuild the conversation from the Runs already stored for this session."""
        history = []
        for run in self.memory.get_all_objects(session_id):
            final = run.get_final_state()
            history.append({"role": "user", "content": final["question"]})
            history.append({"role": "assistant", "content": final["answer"]["answer"]})
        return history

    def ask(self, question: str, session_id: str = "default") -> AgentAnswer:
        print("\n" + "#" * 100)
        print(f"USER [{session_id}]: {question}")
        print("#" * 100)
        self.memory.create_session(session_id)
        initial_state: UdaState = {
            "session_id": session_id, "question": question, "history": self._history(session_id),
            "standalone": question, "local_docs": [], "memory_docs": [],
            "evaluation": None, "web": None, "answer": None,
        }
        run = self.workflow.run(initial_state)
        self.memory.add(run, session_id)           # conversation state for the next question
        self.report(run)
        return AgentAnswer(**run.get_final_state()["answer"])

    def get_session_runs(self, session_id: str = "default") -> List[Run]:
        return self.memory.get_all_objects(session_id)

    # ------------------------------------------------------------------ report
    def report(self, run: Run):
        s = run.get_final_state()
        a = s["answer"]
        print("\n" + "=" * 100)
        print("UDAPLAY REPORT")
        print("=" * 100)
        print(f"Question    : {s['question']}")
        if s["standalone"] != s["question"]:
            print(f"Interpreted : {s['standalone']}")
        print(f"Workflow    : {' -> '.join(snap.step_id for snap in run.snapshots)}")
        print(f"Answer      : {a['answer']}")
        print(f"Confidence  : {a['confidence']}")
        print(f"Source type : {a['source_type']}")
        print("Sources     :")
        for src in a["sources"]:
            print(f"   - {src}")
        print("\nStructured output (JSON):")
        print(json.dumps(a, indent=2, ensure_ascii=False))

### Invoke your agent
One agent object and one session id = one conversation, so it remembers earlier questions. Two of the questions are answered by the local dataset; Mortal Kombat X is not in it, so the web fallback runs.

In [ ]:
# Start with an empty long-term memory so the run clearly shows retrieve -> evaluate -> fallback
# (comment this out to keep what the agent learned in earlier runs)
for _id in memory_collection.get()["ids"]:
    memory_collection.delete(ids=[_id])

agent = UdaPlayAgent()
a1 = agent.ask("When was Pokémon Gold and Silver released?")

In [ ]:
a2 = agent.ask("Which one was the first 3D platformer Mario game?")

In [ ]:
a3 = agent.ask("Was Mortal Kombat X released for PlayStation 5?")

### Conversation state: follow-up question
The next question does not name a game. The agent resolves "it" from the session history.

In [ ]:
a4 = agent.ask("Who published it?")

### Another web-only question

In [ ]:
a5 = agent.ask("What is Rockstar Games working on right now?")

### Session trace
Every question of the session was stored as a `Run`, with one snapshot per step.

In [ ]:
for i, run in enumerate(agent.get_session_runs(), 1):
    s = run.get_final_state()
    path = " -> ".join(snap.step_id for snap in run.snapshots)
    print(f"{i}. {s['question']!r}\n   {path}")

### (Optional) Advanced - Stand Out

**Long-term memory.** Web answers are saved to the persistent `udaplay_memory` collection. A **brand new agent** (empty conversation history) can now recall the Mortal Kombat X answer without searching the web.

In [ ]:
new_session_agent = UdaPlayAgent()
a6 = new_session_agent.ask("Was Mortal Kombat X released for PlayStation 5?", session_id="new-session")

**Also implemented**
- The agent is an explicit **state machine** (`lib.state_machine`) where tools are called from pre-defined steps
- **Structured output**: every answer is an `AgentAnswer` (answer, confidence, source type, sources)
- **Extra tool**: `recall_memory`

### Wrap-up
Before submitting: run every cell, save the notebook with outputs, then **delete or clear your `.env` file**. Keep the `lib/` folder in the submission.